In [1]:
### Cu 003 processing ###


#%% load the packages
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

import defdap.hrdic as hrdic
import defdap.ebsd as ebsd
import defdap.experiment as experiment

from pathlib import Path

import copy 
import pandas as pd
import datetime

from scipy.signal import find_peaks
from scipy.interpolate import griddata

import os

# get dictools stuff 
import sys
# sys.path.append("c:/work/hrdic-tools/")
# import dictools

plt.rcParams['svg.fonttype'] = 'none'

%matplotlib qt

In [8]:
def lsm_read(lsm_file):
    # for reading in lsm output from ZEISS Confomap

    df = pd.read_csv(lsm_file,names=['x','y','z'])

    x = np.asarray(df['x'])
    y = np.asarray(df['y'])
    z = np.asarray(df['z'])

    # calculate shape - this must be done on the raw data 
    x0 = np.nanmin(x)
    x1 = np.nanmax(x)
    y0 = np.nanmin(y)
    y1 = np.nanmax(y)

    x_size = x1 - x0
    y_size = y1 - y0

    # calculate step size 
    x_step = np.round(np.min(np.abs(np.diff(x))),4)
    y_step = np.round(np.max(np.abs(np.diff(y))),4)
    # x_step = np.min(np.abs(np.diff(x)))
    # y_step = np.max(np.abs(np.diff(y)))

    # create new grid to interpolate data onto
    xg,yg = np.meshgrid(np.arange(x0,x1,x_step),np.arange(y0,y1,y_step))



    # remove the weird way that ConfoMaps saves non-measured points
    x = x[z !='***']
    y = y[z !='***']
    z = z[z !='***']


    # interpolate onto grid to produced gridded data
    zg = griddata(np.asarray([x,y]).T,z,(xg,yg),method='nearest')

    return xg, yg, zg 


In [ ]:
# path to LSM file 
lsm_file = './LSM/raw_surface.txt'

# import data
xg,yg,zg = lsm_read(lsm_file)